# 03 - Calibrate ResNet Threshold

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'clean-structure',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
else:
    print('Local run — data/ et work_dirs/ doivent déjà être peuplés.')

→ detect GPU
✓ deps already installed, skipping
✓  /content/INF8225_Projet/data already linked
✓  /content/INF8225_Projet/work_dirs already linked
✓  /content/INF8225_Projet/MedSAM/work_dir already linked
⚠  /content/drive/Shareddrives/Projet_Medsam/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth missing on Drive — skipping /content/INF8225_Projet/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth

Project : /content/INF8225_Projet
Drive   : /content/drive/Shareddrives/Projet_Medsam
Device  : Tesla T4
Torch   : 2.4.0+cu121

⚠  numpy was already loaded in this kernel before setup reinstalled it.
   Runtime → Restart session, then run your imports again
   (no need to rerun setup — deps are pinned on disk).
cwd: /content/INF8225_Projet


In [ ]:
#@title Verify shared assets
from pathlib import Path
from msd_recall_strategy import get_resnet_checkpoint_dir

checkpoint_dir = get_resnet_checkpoint_dir()
required = [
    Path("data/MSD_pancreas/train.json"),
    Path("data/MSD_pancreas/val.json"),
    Path("data/MSD_pancreas/test.json"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("msd_implementation/configs/grounding_dino/pancreas_tumor.py"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"

required += [checkpoint_dir / f"resnet_fold_{i}.pth" for i in range(1, 6)]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"


OK      data/MSD_pancreas/train.json
OK      data/MSD_pancreas/val.json
OK      data/MSD_pancreas/test.json
OK      work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth
OK      work_dirs/tumor_config_v3/tumor_config_v3.py
OK      data/MSD_pancreas/train.json
OK      data/MSD_pancreas/val.json
OK      data/MSD_pancreas/test.json
OK      work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth
OK      work_dirs/tumor_config_v3/tumor_config_v3.py
OK      /content/drive/Shareddrives/Projet_Medsam/resnet_fold_1.pth
OK      /content/drive/Shareddrives/Projet_Medsam/resnet_fold_2.pth
OK      /content/drive/Shareddrives/Projet_Medsam/resnet_fold_3.pth
OK      /content/drive/Shareddrives/Projet_Medsam/resnet_fold_4.pth
OK      /content/drive/Shareddrives/Projet_Medsam/resnet_fold_5.pth


In [ ]:
#@title Run pipeline step
import subprocess
import sys
subprocess.run([sys.executable, "-u", "-m", "msd_implementation.pipelines.resnet18_recall.calibrate_threshold"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', 'calibrate_threshold.py'], returncode=0)

In [ ]:
#@title Inspect threshold artifacts
from pathlib import Path
for p in [Path("outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.txt"), Path("outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.json"), Path("outputs/msd_implementation/resnet18_recall/metrics/calibration_threshold_multi_candidate.csv"), Path("outputs/msd_implementation/resnet18_recall/metrics/threshold_sweep_multi_candidate.csv")]:
    print(("OK     " if p.exists() else "MISSING"), p)
assert Path("outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.txt").exists(), "outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.txt missing"
print("threshold:", Path("outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.txt").read_text().strip())


OK      optimal_threshold.txt
OK      optimal_threshold.json
OK      data/results/calibration_threshold_multi_candidate.csv
OK      data/results/threshold_sweep_multi_candidate.csv
threshold: 0.44


In [ ]:
!cp optimal_threshold.txt /content/drive/Shareddrives/Projet_Medsam/
!cp optimal_threshold.json /content/drive/Shareddrives/Projet_Medsam/